# Chargement des données

In [94]:
import pandas as pd


# ============================================================
# 1. CHARGEMENT DES DATASETS
# ============================================================
 
# Dataset 1 — EliteProspects (infos joueurs)
df_player_dim = pd.read_csv("../Data/player_dim.csv", encoding="latin-1")
 
# Dataset 2 — EliteProspects (stats par saison)
df_player_stats = pd.read_csv("../Data/player_stats.csv", encoding="latin-1")
 
# Dataset 3 — NHL Draft (1963-2022)
df_nhl_draft = pd.read_csv("../Data/nhldraft.csv", encoding="latin-1")

 
print("=== TAILLES INITIALES ===")
print(f"Dataset 1 (player_dim)  : {df_player_dim.shape}")
print(f"Dataset 2 (player_stats): {df_player_stats.shape}")
print(f"Dataset 3 (nhl_draft)   : {df_nhl_draft.shape}")

=== TAILLES INITIALES ===
Dataset 1 (player_dim)  : (109990, 14)
Dataset 2 (player_stats): (402522, 17)
Dataset 3 (nhl_draft)   : (12250, 23)


---

## Enrichissement du dataset player_dim
Nous avons beaucoup de données manquante pour la colonne cible qui contient le rang  du joueur drafté


### Separation des données contenant la variable cible de celle qui n'en n'ont pas

In [95]:

# Dataset avec DRAFT_YEAR non nul
df_dim_drafted = df_player_dim[df_player_dim['DRAFT_OVERALL'].notna()]

# Dataset avec le reste (DRAFT_YEAR nul)
df_dim_not_drafted = df_player_dim[df_player_dim['DRAFT_OVERALL'].isna()]

print(f"Avec DRAFT_OVERALL  : {len(df_dim_drafted)} joueurs")
print(f"Sans DRAFT_OVERALL  : {len(df_dim_not_drafted)} joueurs")


Avec DRAFT_OVERALL  : 1025 joueurs
Sans DRAFT_OVERALL  : 108965 joueurs


## Stratégie 
La stratégie a été de travaillé avec les données du dataset df_dim_not_drafted afin de les enrechir à l'aide du dataset df_nhl_draft

nous cherchons donc à faire la jointure entre ces deux dataset avec comme clé de jointure le prenom+nom que l'on a nommé player et l'année de naissance birthdate

### préparation 

Dans la cellule suivante nous calculons la date de naissance des joueurs en utilisant l'age du joueur et l'année de draft

In [102]:
### Dataset nhl
print(f"le nombre de valeurs manquantes dans la colonne year:{df_nhl_draft['year'].isna().sum()}")
print(f"le nombre de valeurs manquantes dans la colonne age:{df_nhl_draft['age'].isna().sum()}")

# Supprimer les lignes où age est NaN
df_nhl_draft = df_nhl_draft.dropna(subset=['age'])

# Calculer l'année de naissance
# birth_year = année du draft - âge
df_nhl_draft['birth_year'] = (df_nhl_draft['year'] - df_nhl_draft['age']).astype(int)

df_nhl_draft.shape


le nombre de valeurs manquantes dans la colonne year:0
le nombre de valeurs manquantes dans la colonne age:0


(8291, 24)

Dans la cellule suivante nous travaillons avec le dataset df_dim_not_drafted contenant des valeur manquantes dans le rang
- Nous avons supprimés le lignes n'ayant pas de date de naissance
- Ensuite nous avons créé la colonne player contenant les prenom et le nom du joueur concaténés
- Enfin nous avons créé une colonne ne contenant que l'année de naissance du jouer

In [103]:
### player dim

# Supprimer les lignes où la date de naissance est NaN
df_dim_not_drafted = df_dim_not_drafted.dropna(subset=['DATE_OF_BIRTH'])

print(f"Après suppression des NA birthdate : {len(df_dim_not_drafted)} lignes")

# Coller le prénom et le nom en une seule colonne
df_dim_not_drafted['player'] = df_player_dim['FIRST_NAME'] + ' ' + df_dim_not_drafted['LAST_NAME']
#

# Extraire l'année  de naissance depuis la date de naissance complète
df_dim_not_drafted['birth_year'] = pd.to_datetime(df_dim_not_drafted['DATE_OF_BIRTH']).dt.year


df_dim_not_drafted.head()

Après suppression des NA birthdate : 76434 lignes


,ROW_ID,PLAYER_ID,FIRST_NAME,LAST_NAME,DATE_OF_BIRTH,PLACE_OF_BIRTH,NATIONALITY,HEIGHT_CM,WEIGHT_KG,SHOOTS,CONTRACT_THRU,DRAFT_YEAR,DRAFT_ROUND,DRAFT_OVERALL,player,birth_year
0,1,9678.0,Wayne,Gretzky,1961-01-26,"Brantford, ON, CAN",Canada/USA,183.0,84.0,L,Retired,NaN,NaN,NaN,Wayne Gretzky,1961
1,2,21408.0,Mike,Bossy,1957-01-22,"Montréal, QC, CAN",Canada,183.0,84.0,R,Retired,NaN,NaN,NaN,Mike Bossy,1957
2,3,21343.0,Peter,Stastny,1956-09-18,"Bratislava, SVK",Slovakia/Canada,185.0,91.0,L,Retired,NaN,NaN,NaN,Peter Stastny,1956
3,4,42478.0,Dennis,Maruk,1955-11-17,"Toronto, ON, CAN",Canada,173.0,75.0,L,Retired,NaN,NaN,NaN,Dennis Maruk,1955
4,5,22541.0,Bryan,Trottier,1956-07-17,"ValMarie, SK, CAN",Canada/USA,180.0,88.0,L,Retired,NaN,NaN,NaN,Bryan Trottier,1956


Dans la cellule suivante nous avons effectué la jointure entre les deux derniers dataset avec comme clé player et birth_year

In [104]:
df_merged = pd.merge(
    df_dim_not_drafted,
    df_nhl_draft, 
        # a la date de naissance
        # a l'année de draft et l'âge
    on=['player','birth_year'],
    how='left',
    suffixes=('_ep', '_nhl')
)
# Avant
print(f"nombre de lignes : {len(df_merged)} lignes")




nombre de lignes : 76484 lignes


In [84]:
df_merged.columns

Index(['ROW_ID', 'PLAYER_ID', 'FIRST_NAME', 'LAST_NAME', 'DATE_OF_BIRTH',
       'PLACE_OF_BIRTH', 'NATIONALITY', 'HEIGHT_CM', 'WEIGHT_KG', 'SHOOTS',
       'CONTRACT_THRU', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_OVERALL', 'player',
       'birth_year', 'id', 'year', 'overall_pick', 'team', 'nationality',
       'position', 'age', 'to_year', 'amateur_team', 'games_played', 'goals',
       'assists', 'points', 'plus_minus', 'penalties_minutes',
       'goalie_games_played', 'goalie_wins', 'goalie_losses',
       'goalie_ties_overtime', 'save_percentage', 'goals_against_average',
       'point_shares'],
      dtype='object')

## liste des colonnes gardées

In [106]:
cols_a_garder = [
    'ROW_ID', 'PLAYER_ID', 'FIRST_NAME', 'LAST_NAME', 'DATE_OF_BIRTH',
    'PLACE_OF_BIRTH', 'NATIONALITY', 'HEIGHT_CM', 'WEIGHT_KG', 'SHOOTS',
    'CONTRACT_THRU', 'DRAFT_ROUND', 
    'year', 'overall_pick'
]


df_merged = df_merged[cols_a_garder]

df_merged = df_merged.rename(columns={
    'year': 'DRAFT_YEAR',
    'overall_pick': 'DRAFT_OVERALL'
})

df_merged = df_merged.dropna(subset=['DRAFT_YEAR'])

print(df_merged.shape)

(3878, 14)


In [107]:
# Coller deux datasets (empiler les lignes)
df_dim_enrechi = pd.concat([df_merged, df_dim_drafted], axis=0)

print(f"Dataset 1 : {len(df_dim_drafted)} lignes")
print(f"Dataset 2 : {len(df_merged)} lignes")
print(f"Total     : {len(df_dim_enrechi)} lignes")

Dataset 1 : 1025 lignes
Dataset 2 : 3878 lignes
Total     : 4903 lignes


In [108]:
df_dim_enrechi

,ROW_ID,PLAYER_ID,FIRST_NAME,LAST_NAME,DATE_OF_BIRTH,PLACE_OF_BIRTH,NATIONALITY,HEIGHT_CM,WEIGHT_KG,SHOOTS,CONTRACT_THRU,DRAFT_ROUND,DRAFT_YEAR,DRAFT_OVERALL
1,2,21408.0,Mike,Bossy,1957-01-22,"Montréal, QC, CAN",Canada,183.0,84.0,R,Retired,NaN,1977.0,15.0
4,5,22541.0,Bryan,Trottier,1956-07-17,"ValMarie, SK, CAN",Canada/USA,180.0,88.0,L,Retired,NaN,1974.0,22.0
5,6,21482.0,Denis,Savard,1961-02-04,"PointeGatineau, QC, CAN",Canada,178.0,77.0,R,Retired,NaN,1980.0,3.0
6,7,22520.0,Marcel,Dionne,1951-08-03,"Drummondville, QC, CAN",Canada,173.0,84.0,R,Retired,NaN,1971.0,2.0
7,8,32713.0,Bobby,Smith,1958-02-12,"NorthSydney, NS, CAN",Canada,193.0,95.0,L,Retired,NaN,1978.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65407,65408,319079.0,Rasmus,Kupari,2000-03-15,"Kotka, FIN",Finland,186.0,84.0,R,20/21,1.0,2018.0,20.0
65427,65428,396655.0,Kaapo,Kakko,2001-02-13,"Turku, FIN",Finland,187.0,86.0,L,21/22,1.0,2019.0,2.0
65471,65472,428795.0,Ville,Heinola,2001-03-02,"Honkajoki, FIN",Finland,181.0,82.0,L,21/22,1.0,2019.0,20.0
68130,68131,258987.0,Moritz,Seider,2001-04-06,"Zell(Mosel), GER",Germany,192.0,94.0,R,21/22,1.0,2019.0,6.0


In [109]:
for col in df_dim_enrechi.columns:
    print (f'{col} : {df_dim_enrechi[col].isna().sum()} Valeurs manquantes')
    

ROW_ID : 0 Valeurs manquantes
PLAYER_ID : 0 Valeurs manquantes
FIRST_NAME : 0 Valeurs manquantes
LAST_NAME : 0 Valeurs manquantes
DATE_OF_BIRTH : 0 Valeurs manquantes
PLACE_OF_BIRTH : 0 Valeurs manquantes
NATIONALITY : 0 Valeurs manquantes
HEIGHT_CM : 2 Valeurs manquantes
WEIGHT_KG : 2 Valeurs manquantes
SHOOTS : 0 Valeurs manquantes
CONTRACT_THRU : 0 Valeurs manquantes
DRAFT_ROUND : 3888 Valeurs manquantes
DRAFT_YEAR : 10 Valeurs manquantes
DRAFT_OVERALL : 0 Valeurs manquantes


In [31]:
df_dim_enrechi.describe()

,ROW_ID,PLAYER_ID,HEIGHT_CM,WEIGHT_KG,DRAFT_ROUND,DRAFT_YEAR,DRAFT_OVERALL
count,4896.000000,4896.000000,4894.000000,4894.000000,1015.000000,4896.000000,4896.000000
mean,10454.201389,83843.271038,185.426236,89.574377,2.952709,2001.679943,101.617034
std,15389.666943,110545.299670,5.350248,7.532563,1.903420,13.015646,71.238702
min,2.000000,18.000000,163.000000,63.000000,1.000000,1963.000000,1.000000
25%,2114.750000,10992.750000,182.000000,84.000000,1.000000,1992.000000,39.000000
50%,4143.500000,31971.500000,185.000000,89.000000,2.000000,2004.000000,91.000000
75%,9243.500000,107567.750000,189.000000,94.000000,4.000000,2012.000000,156.000000
max,106499.000000,648425.000000,205.000000,120.000000,9.000000,2022.000000,291.000000


In [110]:
import os
file_path = '../Data/df_dim_enrichi.xlsx'
if not os.path.exists(file_path):
    df_dim_enrechi.to_excel(file_path)
    print("Le fichier enregistré")
else:
    print("Le fichier existe déjà")

Le fichier enregistré


---

# Etape de Jointure avec la table des stats

## Calcul du nombre de joueurs ayant des stats disponibles

In [111]:
import duckdb
con = duckdb.connect()

# Requête SQL
query = """
SELECT COUNT(DISTINCT df_dim_enrechi.PLAYER_ID ) AS nb_ids_communs
FROM df_player_stats
INNER JOIN df_dim_enrechi
ON df_dim_enrechi.PLAYER_ID = df_player_stats.PLAYER_ID
"""

result = con.execute(query).fetchdf()

print(result)

   nb_ids_communs
0            4853


In [112]:
# INNER JOIN
query = """
SELECT *
FROM df_player_stats
INNER JOIN df_dim_enrechi
ON df_dim_enrechi.PLAYER_ID = df_player_stats.PLAYER_ID
"""


df_result = con.execute(query).fetchdf()

print(df_result.columns)

Index(['ROW_ID', 'PLAYER_URL', 'PLAYER_ID', 'PLAYER_NAME', 'FIRST_NAME',
       'LAST_NAME', 'PRIMARY_POS', 'SECONDARY_POS', 'G', 'A', 'GP', 'PIM',
       '+/-', 'LEAGUE', 'LEAGUE_YEAR', 'P', 'PPG', 'ROW_ID_1', 'PLAYER_ID_1',
       'FIRST_NAME_1', 'LAST_NAME_1', 'DATE_OF_BIRTH', 'PLACE_OF_BIRTH',
       'NATIONALITY', 'HEIGHT_CM', 'WEIGHT_KG', 'SHOOTS', 'CONTRACT_THRU',
       'DRAFT_ROUND', 'DRAFT_YEAR', 'DRAFT_OVERALL'],
      dtype='object')


In [113]:
df_result.describe()

,ROW_ID,PLAYER_ID,G,A,GP,PIM,+/-,P,PPG,ROW_ID_1,PLAYER_ID_1,HEIGHT_CM,WEIGHT_KG,DRAFT_ROUND,DRAFT_YEAR,DRAFT_OVERALL
count,57917.000000,57917.000000,57814.000000,57812.000000,57771.000000,57389.000000,49761.000000,57812.000000,57263.000000,57917.000000,57917.000000,57901.000000,57901.000000,10614.000000,57903.000000,57917.000000
mean,101957.673101,45258.846522,9.637043,15.599097,44.411417,45.692589,1.070457,25.235038,0.521468,6024.131896,45258.846522,185.580905,90.938965,2.931882,1999.384868,97.795017
std,98921.466176,68470.194163,10.622944,14.691862,24.258965,48.466669,11.545817,24.006889,0.415218,10055.164847,68470.194163,5.239748,7.147424,1.924167,11.101211,72.234424
min,2.000000,18.000000,0.000000,0.000000,0.000000,0.000000,-72.000000,0.000000,0.000000,2.000000,18.000000,163.000000,63.000000,1.000000,1963.000000,1.000000
25%,23317.000000,9142.000000,2.000000,4.000000,26.000000,12.000000,-5.000000,6.000000,0.220000,1910.000000,9142.000000,183.000000,86.000000,1.000000,1991.000000,34.000000
50%,52834.000000,14862.000000,6.000000,12.000000,48.000000,32.000000,0.000000,19.000000,0.437500,3234.000000,14862.000000,185.000000,91.000000,2.000000,2001.000000,84.000000
75%,162205.000000,52326.000000,14.000000,23.000000,65.000000,62.000000,6.000000,37.250000,0.731343,4781.000000,52326.000000,189.000000,95.000000,4.000000,2008.000000,152.000000
max,400532.000000,648425.000000,108.000000,136.000000,93.000000,551.000000,85.000000,234.000000,4.125000,106499.000000,648425.000000,205.000000,120.000000,9.000000,2022.000000,291.000000


### récupération des année de stat avant l'année de draft

In [114]:
df_result['LEAGUE_YEAR_START'] = df_result['LEAGUE_YEAR'].str[:4].astype(int)
stats_avant_draft = df_result[
    df_result['LEAGUE_YEAR_START'] < df_result['DRAFT_YEAR']
]
print (f"le nombre de player distints : {stats_avant_draft['PLAYER_ID'].nunique()}")

le nombre de player distints : 3626


In [116]:
file_path = '../Data/statistiques.xlsx'
if not os.path.exists(file_path):
    stats_avant_draft.to_excel(file_path)
    print("Le fichier enregistré")
else:
    print("Le fichier existe déjà")

Le fichier enregistré
